In [ ]:
import os
import sys
import pandas as pd
from google.colab import userdata

# Install all required libraries
!pip install -q datasets spacy spacy-transformers sentence-transformers flask pyngrok

# Force the system to look for .py files in the current folder
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

print(" Environment setup complete.")

In [ ]:
# The filename as it exists in your GitHub root
DATA_FILE = 'cs_resumes.csv'

if os.path.exists(DATA_FILE):
    print(f" Found {DATA_FILE} in the repository. Loading local copy...")
    cs_df = pd.read_csv(DATA_FILE)
else:
    print(" Data file not found locally. Please ensure 'cs_resumes.csv' was uploaded.")
    # Fallback: if someone runs this without the CSV, it downloads it
    from datasets import load_dataset
    ds = load_dataset("ahmedheakl/resume-atlas", split="train")
    df = ds.to_pandas()
    cs_categories = ['Java Developer', 'Testing', 'Data Science', 'Python Developer', 'Web Designing'] # Example subset
    cs_df = df[df['Category'].isin(cs_categories)].reset_index(drop=True)
    cs_df.to_csv(DATA_FILE, index=False)

print(f"Total resumes loaded: {len(cs_df)}")

In [ ]:
try:
    from parser import clean_resume
    from ner import extract_entities
    from embedder import get_embeddings
    print(" All modules (Parser, NER, Embedder) imported successfully from local files.")
except ImportError as e:
    print(f" Error: A component file is missing! {e}")
    print("Make sure parser.py, ner.py, and embedder.py are uploaded to the same folder.")

In [ ]:
from pyngrok import ngrok

# 1. Authenticate using Colab Secrets
try:
    # Go to the 'Key' icon on the left and add NGROK_AUTH with your token
    NGROK_TOKEN = userdata.get('NGROK_AUTH')
    ngrok.set_auth_token(NGROK_TOKEN)

    # 2. Connect the tunnel
    public_url = ngrok.connect(5000).public_url
    print(f" YOUR APP IS LIVE AT: {public_url}")
    print("Keep this cell running to access the URL.")

    # 3. Run your Flask app
    # (Assuming app.py is uploaded and contains the 'app' object)
    !python app.py

except Exception as e:
    print(f" Connection Error: {e}")
    print("Check if you added 'NGROK_AUTH' to your Colab Secrets.")